In [ ]:
# Cell 1 — install llama.cpp server (prebuilt CUDA binary) and download a small GGUF model
!apt-get update -qq && apt-get install -qq -y curl
!curl -L -o llama-server.tar.gz https://github.com/ggerganov/llama.cpp/releases/latest/download/llama-b1-bin-ubuntu-cuda-cu12.4.1-x64.tar.gz || echo 'DIRECT_BINARY_FAILED_WILL_BUILD_FROM_SOURCE'
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='Qwen/Qwen2.5-1.5B-Instruct-GGUF',
    filename='qwen2.5-1.5b-instruct-q4_k_m.gguf'
)
print('MODEL_PATH:', model_path)

In [ ]:
# Cell 2 — build/install llama.cpp with pip (simpler, portable fallback) and start the server
!pip install -q llama-cpp-python[server]
import subprocess, time
server_proc = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', model_path,
    '--host', '0.0.0.0',
    '--port', '8000',
    '--n_gpu_layers', '-1'
])
time.sleep(20)
print('SERVER_STARTED pid=', server_proc.pid)

In [ ]:
# Cell 3 — start cloudflared tunnel, print the public URL
!curl -L -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, time, re
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break
print()
print('TUNNEL_URL:', url)
print('Keep this cell running — closing it kills the tunnel.')
while True:
    time.sleep(60)